# I'm feeling a little Bowie

## Introdução

Digamos que eu esteja me sentindo de uma maneira muito específica, e eu queira ouvir uma música que consoe meu estado de espírito. Para isso, podemos usar técnicas de Processamento de Linguagem Natural (PLN) para analisar o sentimento de uma frase ou texto e, em seguida, recomendar uma música que corresponda a esse sentimento.

Lógicamente, o universo musical é vasto e diverso, há todavia um artista com tamanho alcance e talento que faz com que todos os outros sejam irrelevantes: David Bowie.

Dessa forma, podemos criar um chatbot que, dada uma descrição do seu estado de espírito, responda a música ideal para enriquecer o seu ser.

## Dados

### Obtenção

Em primeiro lugar é necessário obter todas as letras das músicas de David Bowie. Para tanto, foi utilizado o site [Bowie Wonderworld](https://www.bowiewonderworld.com/songs/dblyrics.htm), que contém todas as letras das músicas do artista. A partir desse site, é possível realizar uma cópia bruta (é necessário desativar os scripts do site para tanto) de todas as letras para um arquivo de texto.

### Transformação e anotação

Primeiramente é necessário transformar o arquivo bruto em um arquivo estruturado, de forma que cada música seja representada por uma linha, contendo o título da música, a letra e o sentimento.

Devido ao altíssimo volume de músicas, a anotação manual de sentimentos para cada música seria inviável.
Por tanto, conclui-se que a a utilização de assistência de IA seria a solução mais apropriada.

A transformação e anotação ocorreram por tanto através do Copilot,
 utilizado o modelo GPT-5.6 Sol, janela de contexto de e raciocínio padrão com o seguinte prompt:


```
I need you to create a csv with the title of each song, the lyrics of each song, and a sentence describing the feelings of the song
```


### Aprimoração

O resultado das anotações iniciais todavia não era satisfatório, sentimentos genéricos e muitas vezes repetitivos foram atribuídos a músicas com sentimentos distintos.

As estratégias adotadas para aprimorar as anotações foram as seguintes:

1. Configurar a janela de contexto para 1.1M tokens
2. Aumentar o nível de raciocínio para "xhigh"
3. Descrever melhor o prompt, incluindo exemplos de sentimentos para músicas específicas. O prompt final utilizado foi o seguinte:

```
I need the feeling field to be more descriptive and individual to each song, for instance, Word on a Wing is a song that evokes a feeling of resignation to a higher power, a plea with God for direction, you can look the net for explanations too
```


## Pré-processamento

### Importando dados

In [1]:
import pandas as pd

lyrics = pd.read_csv("bowie_songs.csv")
lyrics

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
592,Word On A Wing - (1976 Live bonus track),"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
593,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
594,You Better Tell Her,NaN,"Impatient counsel drives the phrase, somebody ..."
595,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


### Limpando dados

In [2]:
lyrics_clean = lyrics.dropna()
lyrics_clean

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
591,Without You I'm Nothing,Strange infatuation seems to grace the evening...,"Sultry self-abasement, decadent images sliding..."
592,Word On A Wing - (1976 Live bonus track),"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
593,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
595,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


In [9]:
_test = [
    ("I feel energized and want to dance with my beloved.", "Let's Dance"),
    ("I'm feeling regretful for falling so low.", "Ashes to Ashes"),
    ("I feel resignation, I just want to understand God's plan for me.", "Word on a Wing"),
    ("I'm completely head over heels in love and want to deliver myself completely", "I would be your slave"),
    ("I'm feeling anxious and stressed about an upcoming event.", "Unknown"),
]

def test(feeling_vectorizer, feeling_vectors, lyrics_vectorizer, lyrics_vectors):
    for phrase, expected in _test:
        most_similar_by_feeling = most_similars(feeling_vectorizer(phrase), feeling_vectors, 1)[0]
        most_similar_by_lyrics = most_similars(lyrics_vectorizer(phrase), lyrics_vectors, 1)[0]
        print(f"Input phrase: {phrase}")
        print(f"\tMost similar songs by feeling: {lyrics_clean.iloc[most_similar_by_feeling]['title']}")
        print(f"\tMost similar songs by lyrics: {lyrics_clean.iloc[most_similar_by_lyrics]['title']}")
        print(f"\tExpected: {expected}")
        print("\n")

## TF-IDF

A primeira abordagem, que servirá de base de comparação para as demais, é a utilização de TF-IDF (Term Frequency-Inverse Document Frequency) para transformar os sentimentos das músicas. Essa técnica permite identificar a importância de cada palavra em relação ao conjunto de documentos (neste caso, os sentimentos das músicas).

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def most_similars(my_feeling_vec, vectors, top_n=5):
    similarities = cosine_similarity(my_feeling_vec, vectors)
    most_similar_indices = similarities.argsort()[0][-top_n:][::-1]
    return most_similar_indices

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

feelings_tfidf_vectorizer = TfidfVectorizer()
feelings_tfidf_matrix = feelings_tfidf_vectorizer.fit_transform(lyrics_clean['feelings'])

lyrics_tfidf_vectorizer = TfidfVectorizer()
lyrics_tfidf_matrix = lyrics_tfidf_vectorizer.fit_transform(lyrics_clean['lyrics'])

In [11]:
test(
    lambda x: feelings_tfidf_vectorizer.transform([x]),
    feelings_tfidf_matrix,
    lambda x: lyrics_tfidf_vectorizer.transform([x]),
    lyrics_tfidf_matrix
)

Input phrase: I feel energized and want to dance with my beloved.
	Most similar songs by feeling: Rupert The Riley
	Most similar songs by lyrics: Let's Dance - (demo)
	Expected: Let's Dance


Input phrase: I'm feeling regretful for falling so low.
	Most similar songs by feeling: I Can't Explain
	Most similar songs by lyrics: As The World Falls Down
	Expected: Ashes to Ashes


Input phrase: I feel resignation, I just want to understand God's plan for me.
	Most similar songs by feeling: Some Weird Sin
	Most similar songs by lyrics: Cygnet Committee
	Expected: Word on a Wing


Input phrase: I'm completely head over heels in love and want to deliver myself completely
	Most similar songs by feeling: I Dig Everything
	Most similar songs by lyrics: And I Say To Myself
	Expected: I would be your slave


Input phrase: I'm feeling anxious and stressed about an upcoming event.
	Most similar songs by feeling: Speed Of Life
	Most similar songs by lyrics: I Took A Trip On A Gemini Spaceship
	Expecte

### Word2Vec

In [12]:
import gensim.downloader as api

# Load pre-trained Word2Vec model
word2vec_model = api.load("word2vec-google-news-300")
word2vec_model["love"]  # Example of getting the vector for a word

array([ 0.10302734, -0.15234375,  0.02587891,  0.16503906, -0.16503906,
        0.06689453,  0.29296875, -0.26367188, -0.140625  ,  0.20117188,
       -0.02624512, -0.08203125, -0.02770996, -0.04394531, -0.23535156,
        0.16992188,  0.12890625,  0.15722656,  0.00756836, -0.06982422,
       -0.03857422,  0.07958984,  0.22949219, -0.14355469,  0.16796875,
       -0.03515625,  0.05517578,  0.10693359,  0.11181641, -0.16308594,
       -0.11181641,  0.13964844,  0.01556396,  0.12792969,  0.15429688,
        0.07714844,  0.26171875,  0.08642578, -0.02514648,  0.33398438,
        0.18652344, -0.20996094,  0.07080078,  0.02600098, -0.10644531,
       -0.10253906,  0.12304688,  0.04711914,  0.02209473,  0.05834961,
       -0.10986328,  0.14941406, -0.10693359,  0.01556396,  0.08984375,
        0.11230469, -0.04370117, -0.11376953, -0.0037384 , -0.01818848,
        0.24316406,  0.08447266, -0.07080078,  0.18066406,  0.03515625,
       -0.09667969, -0.21972656, -0.00328064, -0.03198242,  0.18

In [ ]:
import numpy as np
from nltk.tokenize import word_tokenize
from gensim.models import KeyedVectors

def avg_w2v_vec(text, model: KeyedVectors):
    tokens = word_tokenize(text.lower())
    vectors = [
        model[t]
        for t in tokens
        if t in model
    ]

    if not vectors:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
feelings_word2vec_matrix = [
    avg_w2v_vec(feeling, word2vec_model)
    for feeling in lyrics_clean['feelings']
]
lyrics_word2vec_matrix = [
    avg_w2v_vec(lyric, word2vec_model)
    for lyric in lyrics_clean['lyrics']
]

def most_similars_w2v(my_feeling_vec, vectors, top_n=5):
    similarities = cosine_similarity(my_feeling_vec, vectors)
    most_similar_indices = similarities.argsort()[0][-top_n:][::-1]
    return most_similar_indices

def test_word2vec(model, w2v_matrix):
    for phrase, expected in _test:
        my_feeling_vec = avg_w2v_vec(phrase, model)
        most_similar_indices = most_similars(my_feeling_vec, w2v_matrix, 1)
        print(f"Input phrase: {phrase}")
        print(f"\tMost similar songs: {lyrics_clean.iloc[most_similar_indices[0]]['title']}\n\tExpected: {expected}")
        print("\n")


In [ ]:
def vectorize_with_word2vec(text, model):
    return avg_w2v_vec(text, model).reshape(1, -1)

test(
    vectorize_with_word2vec,
    feelings_word2vec_matrix,
    vectorize_with_word2vec,
    lyrics_word2vec_matrix
)